# Custom Autograd Functions
* Objective: Extend Autograd by writing custom forward and backward passes.
* Task: Sometimes, you need operations that aren't differentiable by default, or you want to optimize memory. Subclass torch.autograd.Function.
* Action:
1. Implement a custom activation function (e.g., Swish or a noisy ReLU).
2. Write the forward static method, saving necessary context using ctx.save_for_backward().
3. Write the backward static method, calculating the chain rule derivative using the saved context and grad_output.
4. Apply your custom function to a tensor and verify gradients using torch.autograd.gradcheck.


1. Implement a custom activation function (e.g., Swish or a noisy ReLU).
2. Write the forward static method, saving necessary context using ctx.save_for_backward().
3. Write the backward static method, calculating the chain rule derivative using the saved context and grad_output.


In [28]:
import torch

class CustomSwish(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        """
        In the forward pass we receive a Tensor containing the input and return
        a Tensor containing the output. ctx is a context object that can be used
        to stash information for backward computation.
        """
        ctx.save_for_backward(x)
        
        output = x * torch.sigmoid(x)
        return output
    
    def backward(ctx, grad_output):
        """
        In the backward pass we receive a Tensor containing the gradient of the loss
        with respect to the output, and we need to compute the gradient of the loss
        with respect to the input.
        """
        
        x, = ctx.saved_tensors
        
        sigmoid_x = torch.sigmoid(x)
        swish_x = x * sigmoid_x
        
        local_grad = sigmoid_x + swish_x * (1.0 - sigmoid_x)
        
        grad_input = grad_output * local_grad
        
        return grad_input

4. Apply your custom function to a tensor and verify gradients using torch.autograd.gradcheck.


In [29]:
from torch.autograd import gradcheck

swish = CustomSwish.apply

input_tensor = torch.randn(3, 3, dtype=torch.double, requires_grad=True)

test_passed = gradcheck(swish, (input_tensor,), eps=1e-6, atol=1e-4)

print(f"Gradient check passed successfully: {test_passed}")

Gradient check passed successfully: True
